In [1]:
from tensorflow.keras.datasets import mnist
from variational_ae import VariationalAutoencoder
import numpy as np
from sklearn.model_selection import train_test_split

2025-08-05 02:23:40.272730: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-05 02:23:40.289292: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754331820.307870   26103 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754331820.312211   26103 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1754331820.325738   26103 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
(x_train_full, _), (x_test, _) = mnist.load_data()
x_all = np.concatenate([x_train_full, x_test], axis=0).astype("float32") / 255.0
x_all = x_all[..., np.newaxis]

x_train, x_val = train_test_split(x_all, test_size=0.05, random_state=42)

print("Train shape:", x_train.shape)
print("Validation shape:", x_val.shape)

Train shape: (66500, 28, 28, 1)
Validation shape: (3500, 28, 28, 1)


In [3]:
LEARNING_RATE = 0.0005
BATCH_SIZE = 32
EPOCHS = 50

In [4]:
input_shape = x_train.shape[1:]
latent_space_dim = 2
decoder_out_filter = 1

In [5]:
# Hyperparameters for the Variational Autoencoder
recon_weight = 1000.0  # Weight for the reconstruction loss.
beta = 1.0  # Weight for the KL divergence loss.

In [6]:
autoencoder = VariationalAutoencoder(input_shape, latent_space_dim, decoder_out_filter, recon_weight, beta, conv_layers_config=[
    {'filters': 32, 'kernel_size': (3, 3), 'strides': (1, 1)},
    {'filters': 64, 'kernel_size': (3, 3), 'strides': (2, 2)},
    {'filters': 64, 'kernel_size': (3, 3), 'strides': (2, 2)},
    {'filters': 64, 'kernel_size': (3, 3), 'strides': (1, 1)}
])

I0000 00:00:1754331823.761296   26103 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9711 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6


In [7]:
autoencoder.compile(learning_rate=LEARNING_RATE)

In [8]:
# autoencoder.summary()

In [9]:
autoencoder.fit(
    x=x_train,
    y={"reconstruction": x_train}, # Autoencoders typically use the same data for input and output
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(x_val, {"reconstruction": x_val}), # Validation data for monitoring
    shuffle=True
)

Epoch 1/50


/home/chua/projects/tf217/lib/python3.12/site-packages/keras/src/optimizers/base_optimizer.py:855: UserWarning: Gradients do not exist for variables ['encoder_conv_layer_1/kernel', 'encoder_batch_norm_layer_1/gamma', 'encoder_batch_norm_layer_1/beta', 'encoder_conv_layer_2/kernel', 'encoder_batch_norm_layer_2/gamma', 'encoder_batch_norm_layer_2/beta', 'encoder_conv_layer_3/kernel', 'encoder_batch_norm_layer_3/gamma', 'encoder_batch_norm_layer_3/beta', 'encoder_conv_layer_4/kernel', 'encoder_batch_norm_layer_4/gamma', 'encoder_batch_norm_layer_4/beta', 'mu/kernel', 'mu/bias', 'log_variance/kernel', 'log_variance/bias', 'decoder_dense_layer/kernel', 'decoder_dense_layer/bias', 'decoder_conv_transpose_layer_1/kernel', 'decoder_batch_norm_layer_1/gamma', 'decoder_batch_norm_layer_1/beta', 'decoder_conv_transpose_layer_2/kernel', 'decoder_batch_norm_layer_2/gamma', 'decoder_batch_norm_layer_2/beta', 'decoder_conv_transpose_layer_3/kernel', 'decoder_batch_norm_layer_3/gamma', 'decoder_batch_

  15/2079 ━━━━━━━━━━━━━━━━━━━━ 23s 12ms/step - kl_loss: 2.9692 - loss: 177.7100 - reconstruction_loss: 0.1747 

I0000 00:00:1754331835.345748   26181 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2079/2079 ━━━━━━━━━━━━━━━━━━━━ 32s 10ms/step - kl_loss: 3.7814 - loss: 59.6195 - reconstruction_loss: 0.0558 - val_kl_loss: 4.7049 - val_loss: 52.6775 - val_reconstruction_loss: 0.0480
Epoch 2/50
2079/2079 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - kl_loss: 4.5244 - loss: 49.8771 - reconstruction_loss: 0.0454 - val_kl_loss: 4.7416 - val_loss: 50.9713 - val_reconstruction_loss: 0.0462
Epoch 3/50
2079/2079 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - kl_loss: 4.7644 - loss: 48.3229 - reconstruction_loss: 0.0436 - val_kl_loss: 4.6727 - val_loss: 47.4473 - val_reconstruction_loss: 0.0428
Epoch 4/50
2079/2079 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - kl_loss: 4.8912 - loss: 47.3406 - reconstruction_loss: 0.0424 - val_kl_loss: 4.8218 - val_loss: 47.2087 - val_reconstruction_loss: 0.0424
Epoch 5/50
2079/2079 ━━━━━━━━━━━━━━━━━━━━ 13s 6ms/step - kl_loss: 4.9525 - loss: 46.7696 - reconstruction_loss: 0.0418 - val_kl_loss: 4.9797 - val_loss: 46.5655 - val_reconstruction_loss: 0.0416
Epoch 6/50
2079/2079 ━━━━━━━━━━━━━━

In [10]:
autoencoder.save_all()  # Save the trained model